# 02 — Extraction Pipeline

**Purpose:** Demonstrate the two-stage extraction pipeline (propose + validate + commit) on a sample story segment, using the live Neo4j graph from Notebook 01.

**Prerequisites:**
- `01_schema_and_ingest.ipynb` has been run (seed graph is populated)
- `.env` configured with `ANTHROPIC_API_KEY`, Neo4j credentials

**Expected outputs:**
- LLM extraction proposals with confidence scores
- Approved vs. flagged split after validation
- Updated graph state after commit
- Edge case demonstration (low confidence, name collision)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv(os.path.join("..", ".env"))

from langchain_anthropic import ChatAnthropic
from src.graph_client import GraphClient
from src.extraction import ExtractionPipeline, NameResolver
from src.schema import ExtractionProposal

In [ ]:
gc = GraphClient(
    uri=os.getenv("NEO4J_URI", "bolt://localhost:7687"),
    user=os.getenv("NEO4J_USER", "neo4j"),
    password=os.getenv("NEO4J_PASSWORD", ""),
)
gc.verify_connectivity()

llm = ChatAnthropic(
    model="claude-sonnet-4-20250514",
    temperature=0.0,
    max_tokens=2048,
)

pipeline = ExtractionPipeline(llm, gc)
print("GraphClient and LLM ready")
print("Existing entities:", gc.get_all_entity_names("main"))

## 1. Stage 1 — Propose

Feed a new story segment through the LLM extraction. This segment introduces a new character (Dorin) and a new location (Ashfall Crossing), plus a relationship change.

In [ ]:
test_segment = (
    "Kael pushed through the tavern door into the cold night air. A figure waited "
    "by the well — Dorin, the blacksmith's apprentice, clutching a rolled parchment. "
    "'She left this for you,' Dorin said, thrusting the scroll into Kael's hands. "
    "'Said you'd know what it means.' Kael unrolled it: a map of Ashfall Crossing, "
    "with Maren's camp marked in red ink. Elara had drawn it before she rode north "
    "alone — against Kael's explicit orders. He crumpled the edge of the parchment. "
    "'That woman will get herself killed.'"
)

existing = gc.get_all_entity_names("main")
proposals = pipeline.propose(test_segment, existing)

print(f"\n--- {len(proposals)} proposals extracted ---\n")
for i, p in enumerate(proposals, 1):
    print(f"{i}. [{p.entity_type}] {p.entity_name} "
          f"(action={p.action}, confidence={p.confidence:.2f})")
    print(f"   Quote: \"{p.supporting_quote}\"")
    if p.properties:
        print(f"   Props: {p.properties}")
    print()

## 2. Stage 2 — Validate

Run deterministic validation: confidence check, fuzzy name resolution, status consistency.

In [ ]:
approved, flagged = pipeline.validate(proposals, "main")

print(f"APPROVED ({len(approved)}):")
for p in approved:
    print(f"  [{p.entity_type}] {p.entity_name} (confidence={p.confidence:.2f}, action={p.action})")

print(f"\nFLAGGED ({len(flagged)}):")
for p in flagged:
    print(f"  [{p.entity_type}] {p.entity_name} (confidence={p.confidence:.2f}, action={p.action})")

## 3. Commit approved proposals

In [ ]:
from src.schema import Segment

seg = Segment(text=test_segment, seq_id=4, branch_id="main")
gc.merge_segment(seg)

committed = pipeline.commit(approved, "main", seq_id=4)
print(f"Committed {committed} of {len(approved)} approved proposals")

print("\nUpdated node counts:", gc.get_node_counts())
print("Updated rel counts:", gc.get_relationship_counts())
print("\nAll entity names:", gc.get_all_entity_names("main"))

## 4. Full pipeline run (propose + validate + commit in one call)

In [ ]:
second_segment = (
    "Dawn broke over Thornwood Bridge. Maren's scouts had spotted riders — "
    "two of them, coming fast from the south. She signaled her archers to hold. "
    "'If it's Kael,' she muttered to her lieutenant, 'let him through. I want "
    "to see his face when he finds the bridge rigged.' The lieutenant — a gaunt "
    "man called Voss — nodded and melted into the treeline."
)

seg2 = Segment(text=second_segment, seq_id=5, branch_id="main")
gc.merge_segment(seg2)

result = pipeline.run(second_segment, branch_id="main", seq_id=5)

print(f"Proposals: {len(result.proposals)}")
print(f"Approved:  {len(result.approved)}")
print(f"Flagged:   {len(result.flagged)}")
print(f"Committed: {result.committed_count}")

print("\nFinal graph state:")
for fact in gc.get_graph_summary_facts("main"):
    print(f"  - {fact}")

## 5. Edge case: name fuzzy matching demo

In [ ]:
existing = gc.get_all_entity_names("main")
print("Existing names:", existing)

test_cases = ["Kaeel", "Ellara", "Marren", "Dorin", "Completely New Name"]
for name in test_cases:
    match = NameResolver.fuzzy_match(name, existing)
    print(f"  '{name}' -> {f'matched: {match!r}' if match else 'no match (new entity)'}")

In [ ]:
gc.close()
print("Done. GraphClient connection closed.")